In [34]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import numpy as np
from math import fabs,cos,asin,sqrt
from tqdm import tqdm
from pyproj import Geod
from datetime import datetime

In [ ]:

import Files.usgs_dataparser as usgs_dataparser

eqcatalogue_df = usgs_dataparser.usgs_dataparser(r'Files\USGS_Turkiye_Deprem_Listesi.csv')
eqcatalogue_gdf = gpd.GeoDataFrame(
    eqcatalogue_df, 
    geometry=gpd.points_from_xy(eqcatalogue_df['longitude'], eqcatalogue_df['latitude'],crs="EPSG:4326")
)

polyzones_gdf = gpd.read_file(r'Files\TR_AREA_02.05.2023.kml', driver='KML')
polyzones_gdf['zone_id'] = range(1, len(polyzones_gdf) + 1)
#polyzones_gdf = eqcatalogue_gdf[.to_crs(polyzones_gdf.crs)
polyzones_gdf = pd.concat([polyzones_gdf[~polyzones_gdf['zone_id'].isin([102, 7])]
                           ,gpd.GeoDataFrame({'zone_id': 999,'Name': 'merged_zone','Description': ' ', 'geometry': polyzones_gdf[polyzones_gdf['zone_id'] == 7].union(polyzones_gdf[polyzones_gdf['zone_id'] == 102],align=False)})])
zonedeqcatalogue = gpd.sjoin(eqcatalogue_gdf, polyzones_gdf, how='left', predicate='within')
print(zonedeqcatalogue.sindex.valid_query_predicates)
zonedeqcatalogue.drop(columns=['index_right','Name','Description'],inplace=True)
unzonedeqcatalogue = zonedeqcatalogue[zonedeqcatalogue['zone_id'].isnull()]
zonedeqcatalogue.drop(zonedeqcatalogue[zonedeqcatalogue['zone_id'].isnull()].index,inplace=True)

def find_nearest_polyzone(point, polyzones):
    polyzones = polyzones.geometry.tolist()
    nearest = None
    min_distance = float('inf')
    for polyzone in polyzones:
        distance = point.distance(polyzone)
        if distance < min_distance:
            min_distance = distance
            nearest = polyzone
            nearest_id = polyzones.index(polyzone)+1
    return nearest_id

nearest_polyzones = []
for x in unzonedeqcatalogue.loc[pd.isna(unzonedeqcatalogue["zone_id"]), :].index:
    nearest_polyzone = find_nearest_polyzone(unzonedeqcatalogue.loc[x,'geometry'], polyzones_gdf)
    unzonedeqcatalogue.loc[x,'zone_id'] = nearest_polyzone

zonedeqcatalogue['zone_id'],unzonedeqcatalogue['zone_id'] = pd.to_numeric(zonedeqcatalogue['zone_id'],downcast='integer'),pd.to_numeric(unzonedeqcatalogue['zone_id'],downcast='integer')
fullzonedeqcatalogue = pd.concat([zonedeqcatalogue,unzonedeqcatalogue]).sort_index()

display(fullzonedeqcatalogue)



Raw earthquake catalogue from USGS


,time,latitude,longitude,depth,mag,...,magError,magNst,status,locationSource,magSource
2510,2008-09-28T11:32:12.000Z,36.662,30.074,63.7,4.0,...,NaN,NaN,reviewed,isk,isk
2661,2007-08-17T21:55:34.300Z,38.680,25.230,39.0,4.2,...,NaN,NaN,reviewed,ath,ath
4124,1992-06-30T20:22:06.550Z,38.185,30.084,10.0,4.1,...,NaN,NaN,reviewed,us,dda
4568,1988-08-20T06:52:30.220Z,35.730,31.215,10.0,4.0,...,NaN,NaN,reviewed,us,hlw
4569,1988-08-19T18:50:27.870Z,37.904,29.081,10.0,4.3,...,NaN,NaN,reviewed,us,hlw
4571,1988-08-15T07:47:09.320Z,37.924,29.344,11.1,4.2,...,NaN,NaN,reviewed,us,hlw
4580,1988-07-22T08:21:58.200Z,36.712,31.568,127.3,4.0,...,NaN,NaN,reviewed,us,hlw
4582,1988-07-17T21:29:06.900Z,36.235,27.954,33.0,4.1,...,NaN,NaN,reviewed,us,hlw


['mwr' 'mb' 'mww' 'ml' 'mwb' 'mw' 'mwc' 'mblg' 'm' 'md' 'ms']
['mw' 'mb' 'ml' 'm' 'md' 'ms']
Earthquake catalogue after manipulation


,ID,date_decimal,year,month,day,latitude,longitude,depth,mag,place
0,0,2024.599772,2024,8,7,37.6819,35.9702,10.000,4.60,"15 km SSE of Feke, Turkey"
1,1,2024.583333,2024,8,1,38.1437,38.2237,10.000,4.46,"13 km N of Çelikhan, Turkey"
2,2,2024.568493,2024,7,26,35.6464,25.6524,10.000,4.77,"39 km NNE of Sísion, Greece"
4,4,2024.557534,2024,7,22,41.5032,45.9528,31.311,4.56,"13 km S of Ts’nori, Georgia"
3,3,2024.557534,2024,7,22,39.6736,26.3745,11.394,4.50,"8 km NNW of Ayvacık, Turkey"
...,...,...,...,...,...,...,...,...,...,...
6088,6088,1931.030137,1931,1,12,37.9720,31.6190,15.000,5.32,"2 km NE of Hüyük, Turkey"
6089,6089,1931.030137,1931,1,12,38.0330,31.7490,15.000,5.57,"14 km SSE of Do?anhisar, Turkey"
6090,6090,1930.941324,1930,12,10,39.9720,39.1520,15.000,5.81,"24 km S of ?iran, Turkey"
6091,6091,1930.815753,1930,10,25,39.0320,45.5290,15.000,5.30,"12 km NW of Culfa, Azerbaijan"


{None, 'contains_properly', 'contains', 'covered_by', 'overlaps', 'touches', 'covers', 'dwithin', 'intersects', 'within', 'crosses'}


c:\Users\mts00029\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,ID,date_decimal,year,month,day,...,depth,mag,place,geometry,zone_id
0,0,2024.599772,2024,8,7,...,10.000,4.60,"15 km SSE of Feke, Turkey",POINT (35.9702 37.6819),10
1,1,2024.583333,2024,8,1,...,10.000,4.46,"13 km N of Çelikhan, Turkey",POINT (38.2237 38.1437),999
2,2,2024.568493,2024,7,26,...,10.000,4.77,"39 km NNE of Sísion, Greece",POINT (25.6524 35.6464),22
3,3,2024.557534,2024,7,22,...,11.394,4.50,"8 km NNW of Ayvacık, Turkey",POINT (26.3745 39.6736),18
4,4,2024.557534,2024,7,22,...,31.311,4.56,"13 km S of Ts’nori, Georgia",POINT (45.9528 41.5032),46
...,...,...,...,...,...,...,...,...,...,...,...
6088,6088,1931.030137,1931,1,12,...,15.000,5.32,"2 km NE of Hüyük, Turkey",POINT (31.619 37.972),38
6089,6089,1931.030137,1931,1,12,...,15.000,5.57,"14 km SSE of Do?anhisar, Turkey",POINT (31.749 38.033),38
6090,6090,1930.941324,1930,12,10,...,15.000,5.81,"24 km S of ?iran, Turkey",POINT (39.152 39.972),2
6091,6091,1930.815753,1930,10,25,...,15.000,5.30,"12 km NW of Culfa, Azerbaijan",POINT (45.529 39.032),50


In [ ]:
def calc_windows(fMagnitude, nMethod=1):
    """Calculate window lengths in space and time for declustering"""
    if nMethod in [1, 2]:  # Gardner & Knopoff (1974) base formula
        fSpace = 10**(0.1238 * fMagnitude + 0.983)
        fTime = np.where(
            fMagnitude >= 6.5,
            (10**(0.032 * fMagnitude + 2.7389)) / 365,
            (10**(0.5409 * fMagnitude - 0.547)) / 365
        )
    else:
        raise ValueError("Implemented methods: 1-GK74, 2-ZoneAdjusted")
    return fSpace, fTime

def decluster_catalog(fullzonedeqcatalogue, nMethod=1):
    # Preprocessing
    mCatalog = fullzonedeqcatalogue.drop(['depth'], axis=1).copy()
    mCatalog = mCatalog.sort_values('date_decimal').reset_index(drop=True)
    
    # Initialize arrays
    nEvents = len(mCatalog)
    vCluster = np.zeros(nEvents, dtype=int)
    vMainCluster = -np.ones(nEvents, dtype=int)
    vShockType = np.full(nEvents, 'MS', dtype='U2')
    vZones = mCatalog['zone_id'].values
    vDepShocksMainZones = np.zeros(nEvents, dtype=int)
    vSameZone = np.zeros(nEvents, dtype=int)
    
    # Geospatial setup
    gdf = gpd.GeoDataFrame(
        mCatalog,
        geometry=gpd.points_from_xy(mCatalog.longitude, mCatalog.latitude),
        crs="EPSG:4326"
    )
    geod = Geod(ellps='WGS84')
    print("Calculating distance matrix...")
    
    # Precompute all pairwise distances (corrected)
    lons = gdf.geometry.x.to_numpy()
    lats = gdf.geometry.y.to_numpy()
    distances = np.zeros((nEvents, nEvents))
    
    for i in tqdm(range(nEvents)):
        # Create arrays with matching lengths
        lons1 = np.full(nEvents, lons[i])
        lats1 = np.full(nEvents, lats[i])
        _, _, dist = geod.inv(lons1, lats1, lons, lats)
        distances[i] = dist / 1000  # Convert to km
    
    # Main declustering loop
    print("Processing clusters...")
    for nEvent in tqdm(range(nEvents)):
        if vCluster[nEvent] != 0:
            continue  # Skip already clustered events
            
        current_mag = mCatalog.loc[nEvent, 'mag']
        current_zone = mCatalog.loc[nEvent, 'zone_id']
        current_time = mCatalog.loc[nEvent, 'date_decimal']
        mainshock_idx = nEvent
        cluster_members = []
        
        while True:
            # Determine zone relationships for current mainshock
            same_zone = (vZones == current_zone)
            if nMethod == 2:
                adj_mag = np.where(same_zone, current_mag, current_mag - 1)
            else:
                adj_mag = np.full(nEvents, current_mag)
            
            # Calculate windows for all events
            fSpace, fTime = calc_windows(adj_mag, nMethod=1)
            time_end = current_time + fTime
            
            # Find potential cluster members
            valid = (vCluster == 0) & \
                    (distances[mainshock_idx] <= fSpace) & \
                    (mCatalog['date_decimal'] >= current_time) & \
                    (mCatalog['date_decimal'] <= time_end)
            
            new_members = np.where(valid)[0].tolist()
            
            # Check for larger magnitude in new members
            if not new_members:
                break
            max_mag = mCatalog.loc[new_members, 'mag'].max()
            if max_mag <= current_mag:
                break  # Cluster finalized
                
            # Find new mainshock (first occurrence of max mag)
            new_main = mCatalog.loc[new_members, 'mag'].idxmax()
            current_mag = mCatalog.loc[new_main, 'mag']
            current_zone = mCatalog.loc[new_main, 'zone_id']
            mainshock_idx = new_main
        
        # Finalize cluster assignments
        if len(new_members) > 0:
            cluster_id = vCluster.max() + 1
            vCluster[new_members] = cluster_id
            vCluster[mainshock_idx] = 0  # Set mainshock
            vMainCluster[new_members] = mainshock_idx
            vDepShocksMainZones[new_members] = current_zone
            vSameZone[new_members] = (vZones[new_members] == current_zone).astype(int)
            vShockType[new_members] = np.where(
                mCatalog.loc[new_members, 'date_decimal'] > current_time,
                'AS', 'FS'
            )
    # Post-processing
    vShockType[vCluster == 0] = 'MS'
    mCatalog = mCatalog.assign(
        cluster_id=vCluster,
        shock_type=vShockType,
        mainshock_zone=vDepShocksMainZones,
        same_zone=vSameZone
    )
    return {
        'declustered': mCatalog[vCluster == 0],
        'aftershocks': mCatalog[vCluster > 0],
        'full_catalog': mCatalog,
        'cluster_numbers': vCluster,
        'main_cluster_numbers': vMainCluster
    }

In [4]:
# Run with default method (GK74)
result_GK74 = decluster_catalog(fullzonedeqcatalogue, nMethod=1)

# Run with zone-adjusted method
result_proposed = decluster_catalog(fullzonedeqcatalogue, nMethod=2)

# Access results
declustered_GK74 = result_GK74['declustered']
aftershocks_GK74 = result_GK74['aftershocks']
fullcatalog_GK74 = result_GK74['full_catalog']

declustered_proposed = result_proposed['declustered']
aftershocks_proposed = result_proposed['aftershocks']
fullcatalog_proposed = result_proposed['full_catalog']

Calculating distance matrix...


100%|██████████| 6093/6093 [00:35<00:00, 170.39it/s]


Processing clusters...


100%|██████████| 6093/6093 [00:06<00:00, 984.53it/s] 


Calculating distance matrix...


100%|██████████| 6093/6093 [00:35<00:00, 169.96it/s]


Processing clusters...


100%|██████████| 6093/6093 [00:07<00:00, 856.28it/s] 


In [5]:
fullcatalog_GK74.to_excel(r'Files\finalcatalog_GK74.xlsx', index=False)
fullcatalog_proposed.to_excel(r'Files\finalcatalog_Proposed.xlsx', index=False)

In [33]:

finalcatalog_proposed = pd.read_excel('Files/finalcatalog_Proposed.xlsx')
finalcatalog_GK74 = pd.read_excel('Files/finalcatalog_GK74.xlsx')

display(finalcatalog_GK74[['ID','date_decimal','latitude','longitude','mag','place','zone_id','shock_type','mainshock_zone']].sort_values('mag',ascending=False).head(7).to_latex())
display(finalcatalog_proposed[['ID','date_decimal','latitude','longitude','mag','place','zone_id','shock_type','mainshock_zone']].sort_values('mag',ascending=False).head(7).to_latex())


'\\begin{tabular}{lrrrrrlrlr}\n\\toprule\n & ID & date_decimal & latitude & longitude & mag & place & zone_id & shock_type & mainshock_zone \\\\\n\\midrule\n83 & 6009 & 1939.985160 & 39.907000 & 39.586000 & 7.800000 & 20 km NNE of Erzincan, Turkey & 2 & MS & 2 \\\\\n5479 & 754 & 2023.097032 & 37.225600 & 37.014300 & 7.800000 & Pazarcik earthquake, Kahramanmaras earthquake sequence & 999 & MS & 999 \\\\\n224 & 5870 & 1956.521918 & 36.664000 & 25.957000 & 7.700000 & 19 km SSE of Amorgós, Greece & 70 & MS & 70 \\\\\n132 & 5960 & 1944.083333 & 40.660000 & 32.998000 & 7.600000 & 10 km WNW of Orta, Turkey & 15 & MS & 15 \\\\\n2535 & 3580 & 1999.627169 & 40.748000 & 29.864000 & 7.600000 & 4 km ESE of Derince, Turkey & 8 & MS & 8 \\\\\n5525 & 651 & 2023.097032 & 38.010600 & 37.196200 & 7.500000 & Elbistan earthquake, Kahramanmaras earthquake sequence & 10 & AS & 999 \\\\\n129 & 5963 & 1943.901826 & 40.867000 & 33.651000 & 7.500000 & 6 km SSE of Ilgaz, Turkey & 2 & FS & 15 \\\\\n\\bottomrule\n\

'\\begin{tabular}{lrrrrrlrlr}\n\\toprule\n & ID & date_decimal & latitude & longitude & mag & place & zone_id & shock_type & mainshock_zone \\\\\n\\midrule\n83 & 6009 & 1939.985160 & 39.907000 & 39.586000 & 7.800000 & 20 km NNE of Erzincan, Turkey & 2 & MS & 2 \\\\\n5479 & 754 & 2023.097032 & 37.225600 & 37.014300 & 7.800000 & Pazarcik earthquake, Kahramanmaras earthquake sequence & 999 & MS & 999 \\\\\n224 & 5870 & 1956.521918 & 36.664000 & 25.957000 & 7.700000 & 19 km SSE of Amorgós, Greece & 70 & MS & 70 \\\\\n132 & 5960 & 1944.083333 & 40.660000 & 32.998000 & 7.600000 & 10 km WNW of Orta, Turkey & 15 & MS & 15 \\\\\n2535 & 3580 & 1999.627169 & 40.748000 & 29.864000 & 7.600000 & 4 km ESE of Derince, Turkey & 8 & MS & 8 \\\\\n5525 & 651 & 2023.097032 & 38.010600 & 37.196200 & 7.500000 & Elbistan earthquake, Kahramanmaras earthquake sequence & 10 & MS & 10 \\\\\n129 & 5963 & 1943.901826 & 40.867000 & 33.651000 & 7.500000 & 6 km SSE of Ilgaz, Turkey & 2 & FS & 15 \\\\\n\\bottomrule\n\\

In [31]:
finalcatalog_GK74[['ID','date_decimal','latitude','longitude','mag','place','zone_id','shock_type','mainshock_zone']].sort_values('mag',ascending=False).head(7)

,ID,date_decimal,latitude,longitude,mag,place,zone_id,shock_type,mainshock_zone
83,6009,1939.985160,39.9070,39.5860,7.8,"20 km NNE of Erzincan, Turkey",2,MS,2
5479,754,2023.097032,37.2256,37.0143,7.8,"Pazarcik earthquake, Kahramanmaras earthquake ...",999,MS,999
224,5870,1956.521918,36.6640,25.9570,7.7,"19 km SSE of Amorgós, Greece",70,MS,70
132,5960,1944.083333,40.6600,32.9980,7.6,"10 km WNW of Orta, Turkey",15,MS,15
2535,3580,1999.627169,40.7480,29.8640,7.6,"4 km ESE of Derince, Turkey",8,MS,8
5525,651,2023.097032,38.0106,37.1962,7.5,"Elbistan earthquake, Kahramanmaras earthquake ...",10,AS,999
129,5963,1943.901826,40.8670,33.6510,7.5,"6 km SSE of Ilgaz, Turkey",2,FS,15


In [32]:
finalcatalog_proposed[['ID','date_decimal','latitude','longitude','mag','place','zone_id','shock_type','mainshock_zone']].sort_values('mag',ascending=False).head(7)

,ID,date_decimal,latitude,longitude,mag,place,zone_id,shock_type,mainshock_zone
83,6009,1939.985160,39.9070,39.5860,7.8,"20 km NNE of Erzincan, Turkey",2,MS,2
5479,754,2023.097032,37.2256,37.0143,7.8,"Pazarcik earthquake, Kahramanmaras earthquake ...",999,MS,999
224,5870,1956.521918,36.6640,25.9570,7.7,"19 km SSE of Amorgós, Greece",70,MS,70
132,5960,1944.083333,40.6600,32.9980,7.6,"10 km WNW of Orta, Turkey",15,MS,15
2535,3580,1999.627169,40.7480,29.8640,7.6,"4 km ESE of Derince, Turkey",8,MS,8
5525,651,2023.097032,38.0106,37.1962,7.5,"Elbistan earthquake, Kahramanmaras earthquake ...",10,MS,10
129,5963,1943.901826,40.8670,33.6510,7.5,"6 km SSE of Ilgaz, Turkey",2,FS,15
